In [5]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, recall_score, precision_score
import joblib
# === Step 1: Load and preprocess data ===

df = pd.read_csv("full_data_unhealthy_imputed.csv")

# Ensure hour column is numeric and data sorted properly
df['hour'] = pd.to_numeric(df['hour'], errors='coerce')
df = df.sort_values(['cow', 'date', 'hour'])

# === Step 2: Feature Engineering ===

# Basic features you have
feature_cols = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'disturbance']

# Add engineered features:
# 1) Eat to Rest ratio
df['eat_rest_ratio'] = df['EAT'] / (df['REST'] + 1e-3)  # avoid division by zero

# 2) Delta features: change compared to previous hour, per cow
df['delta_eat'] = df.groupby('cow')['EAT'].diff().fillna(0)
df['delta_rest'] = df.groupby('cow')['REST'].diff().fillna(0)
df['delta_activity'] = df.groupby('cow')['ACTIVITY_LEVEL'].diff().fillna(0)

# Append engineered features to feature set
engineered_features = ['eat_rest_ratio', 'delta_eat', 'delta_rest', 'delta_activity']
feature_cols += engineered_features

# === Step 3: Targets ===
target_cols = ['oestrus', 'calving', 'lameness', 'mastitis']

# === Step 4: Normalize features ===
scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

# === Step 5: Create sequences ===

def create_sequences(data, features, targets, seq_len=24):
    X, Y = [], []
    for _, group in data.groupby('cow'):
        group = group.sort_values('hour')
        # Skip sequences shorter than seq_len
        if len(group) < seq_len:
            continue
        for i in range(len(group) - seq_len + 1):
            X.append(group.iloc[i:i+seq_len][features].values)
            Y.append(group.iloc[i+seq_len-1][targets].values)  # label at last timestep
    return np.array(X), np.array(Y)

X, Y = create_sequences(df, feature_cols, target_cols, seq_len=24)
# Ensure arrays are numeric with correct dtype
X = np.array(X, dtype=np.float32)
Y = np.array(Y, dtype=np.float32)
print(f"Data shapes - X: {X.shape}, Y: {Y.shape}")

# === Step 6: Train-test split ===
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                        torch.tensor(Y_train, dtype=torch.float32))

val_ds = TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                      torch.tensor(Y_val, dtype=torch.float32))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128)

# === Step 7: Define the TCN model ===

class TCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size,
                              dilation=dilation, padding=padding)
        self.relu = nn.ReLU()
        self.bn = nn.BatchNorm1d(out_channels)

    def forward(self, x):
        x = self.conv(x)
        x = self.relu(x)
        x = self.bn(x)
        return x

class TCNModel(nn.Module):
    def __init__(self, num_features, num_labels, num_channels=[64, 64]):
        super().__init__()
        layers = []
        in_channels = num_features
        for out_ch in num_channels:
            layers.append(TCNBlock(in_channels, out_ch, kernel_size=3))
            in_channels = out_ch
        self.network = nn.Sequential(*layers)
        self.head = nn.Linear(in_channels, num_labels)

    def forward(self, x):
        # x shape: [B, T, F] -> [B, F, T]
        x = x.permute(0, 2, 1)
        x = self.network(x)
        # Take features at last time step
        x = x[:, :, -1]
        return self.head(x)  # raw logits, no sigmoid here

# === Step 8: Prepare loss, optimizer, and device ===

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TCNModel(num_features=len(feature_cols), num_labels=len(target_cols)).to(device)

# Compute positive class weights: inverse frequency per label
pos_freq = Y_train.sum(axis=0) / len(Y_train)
pos_weight = torch.tensor((1.0 / (pos_freq + 1e-6)), dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# === Step 9: Training loop ===

num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)

    avg_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{num_epochs}, Training Loss: {avg_loss:.5f}")

# === Step 10: Evaluate model and tune thresholds ===

model.eval()
all_preds = []
all_true = []

with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device)
        preds = model(xb)
        preds = torch.sigmoid(preds).cpu().numpy()
        all_preds.append(preds)
        all_true.append(yb.numpy())

all_preds = np.concatenate(all_preds)
all_true = np.concatenate(all_true)

# Find optimal thresholds for each label by maximizing F1 on validation set
optimal_thresholds = []

print("\nThreshold tuning and metrics per disease:")

for i, disease in enumerate(target_cols):
    best_f1 = 0
    best_thresh = 0.5
    for thresh in np.arange(0.01, 0.60, 0.01):
        preds_bin = (all_preds[:, i] > thresh).astype(int)
        f1 = f1_score(all_true[:, i], preds_bin, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh
    optimal_thresholds.append(best_thresh)
    preds_bin = (all_preds[:, i] > best_thresh).astype(int)
    rec = recall_score(all_true[:, i], preds_bin, zero_division=0)
    prec = precision_score(all_true[:, i], preds_bin, zero_division=0)
    print(f"{disease}: F1={best_f1:.4f}, Recall={rec:.4f}, Precision={prec:.4f}, Threshold={best_thresh:.2f}")

print("\nUse these thresholds for your predictions.")

# === Optional: Save the model and scaler ===
# Save PyTorch model weights
torch.save(model.state_dict(), 'tcn_cattle_model.pth')

# Save the scaler using joblib
import joblib
joblib.dump(scaler, 'feature_scaler.save')

print("Model and scaler saved successfully.")


# Re-create the model instance exactly as before
model = TCNModel(num_features=len(feature_cols), num_labels=len(target_cols))
model.load_state_dict(torch.load('tcn_cattle_model.pth'))
model.eval()  # for inference

# Load the scaler for feature transformations
scaler = joblib.load('feature_scaler.save')

print("Model and scaler loaded successfully.")


Data shapes - X: (1929250, 24, 9), Y: (1929250, 4)
Epoch 1/20, Training Loss: 1.18576
Epoch 2/20, Training Loss: 1.16004
Epoch 3/20, Training Loss: 1.15513
Epoch 4/20, Training Loss: 1.14795
Epoch 5/20, Training Loss: 1.14409
Epoch 6/20, Training Loss: 1.14115
Epoch 7/20, Training Loss: 1.13762
Epoch 8/20, Training Loss: 1.13522
Epoch 9/20, Training Loss: 1.13025
Epoch 10/20, Training Loss: 1.13162
Epoch 11/20, Training Loss: 1.12619
Epoch 12/20, Training Loss: 1.12509
Epoch 13/20, Training Loss: 1.12449
Epoch 14/20, Training Loss: 1.12235
Epoch 15/20, Training Loss: 1.12135
Epoch 16/20, Training Loss: 1.11870
Epoch 17/20, Training Loss: 1.11632
Epoch 18/20, Training Loss: 1.11572
Epoch 19/20, Training Loss: 1.11147
Epoch 20/20, Training Loss: 1.11018

Threshold tuning and metrics per disease:
oestrus: F1=0.0263, Recall=0.3499, Precision=0.0137, Threshold=0.59
calving: F1=0.0396, Recall=0.6503, Precision=0.0204, Threshold=0.59
lameness: F1=0.0143, Recall=0.6144, Precision=0.0072, Thres

In [9]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, recall_score, precision_score

# === Step 1: Load data ===
df = pd.read_csv("full_data_unhealthy_imputed.csv")
df['hour'] = pd.to_numeric(df['hour'], errors='coerce')
df = df.sort_values(['cow', 'date', 'hour'])

# === Step 2: Feature Engineering ===
# Base features
base_features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'disturbance']

# Engineered features
df['eat_rest_ratio'] = df['EAT'] / (df['REST'] + 1e-3)
df['delta_eat'] = df.groupby('cow')['EAT'].diff().fillna(0)
df['delta_rest'] = df.groupby('cow')['REST'].diff().fillna(0)
df['delta_activity'] = df.groupby('cow')['ACTIVITY_LEVEL'].diff().fillna(0)

feature_cols = base_features + ['eat_rest_ratio', 'delta_eat', 'delta_rest', 'delta_activity']

target_cols = ['oestrus', 'calving', 'lameness', 'mastitis']

# === Step 3: Normalize features ===
scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

# === Step 4: Sequence creation ===
def create_sequences(data, features, targets, seq_len=24):
    X, Y = [], []
    for _, group in data.groupby('cow'):
        group = group.sort_values('hour')
        if len(group) < seq_len:
            continue
        for i in range(len(group) - seq_len + 1):
            X.append(group.iloc[i:i+seq_len][features].values)
            Y.append(group.iloc[i+seq_len-1][targets].values)
    return np.array(X, dtype=np.float32), np.array(Y, dtype=np.float32)

X, Y = create_sequences(df, feature_cols, target_cols, seq_len=24)
print(f"Sequence data shapes: X={X.shape}, Y={Y.shape}")

# === Step 5: Train-test split ===
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(Y_train))
val_ds = TensorDataset(torch.tensor(X_val), torch.tensor(Y_val))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=4)
val_loader = DataLoader(val_ds, batch_size=128, num_workers=4)

# === Step 6: Define Focal Loss for multi-label ===
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, pos_weight=None, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.pos_weight = pos_weight
        self.reduction = reduction

    def forward(self, inputs, targets):
        # inputs: logits; targets: 0/1 labels
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, pos_weight=self.pos_weight, reduction='none')
        probas = torch.sigmoid(inputs)
        pt = probas * targets + (1 - probas) * (1 - targets)
        focal_factor = (1 - pt) ** self.gamma
        loss = self.alpha * focal_factor * bce_loss

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss

# === Step 7: Define TCN model ===
class TCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, dilation=dilation, padding=padding)
        self.relu = nn.ReLU()
        self.bn = nn.BatchNorm1d(out_channels)

    def forward(self, x):
        x = self.conv(x)
        x = self.relu(x)
        x = self.bn(x)
        return x

class TCNModel(nn.Module):
    def __init__(self, num_features, num_labels, num_channels=[64, 64]):
        super().__init__()
        layers = []
        in_channels = num_features
        for out_ch in num_channels:
            layers.append(TCNBlock(in_channels, out_ch))
            in_channels = out_ch
        self.network = nn.Sequential(*layers)
        self.head = nn.Linear(in_channels, num_labels)

    def forward(self, x):
        # x shape: [B, T, F] -> [B, F, T]
        x = x.permute(0,2,1)
        x = self.network(x)
        x = x[:,:,-1]
        return self.head(x)  # logits

# === Step 8: Setup device, model, loss, optimizer ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TCNModel(num_features=len(feature_cols), num_labels=len(target_cols)).to(device)

# Calculate pos_weight as inverse frequency for focal loss (convert to tensor on device)
pos_freq = Y_train.sum(axis=0) / Y_train.shape[0]
pos_weight = torch.tensor(1.0 / (pos_freq + 1e-6), dtype=torch.float32).to(device)

criterion = FocalLoss(alpha=1, gamma=2, pos_weight=pos_weight, reduction='mean')
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# === Step 9: Training loop ===
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    avg_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{num_epochs} - Training Loss: {avg_loss:.5f}")

# === Step 10: Evaluation and Threshold tuning ===
model.eval()
all_preds = []
all_true = []

with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device)
        logits = model(xb)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_preds.append(probs)
        all_true.append(yb.numpy())

all_preds = np.concatenate(all_preds)
all_true = np.concatenate(all_true)

# Tune thresholds per label by maximizing F1-score
optimal_thresholds = []

print("\nPer-disease threshold tuning and metrics:")

for i, disease in enumerate(target_cols):
    best_f1 = 0
    best_thresh = 0.5
    for thresh in np.arange(0.01, 0.60, 0.01):
        bin_preds = (all_preds[:, i] > thresh).astype(int)
        f1 = f1_score(all_true[:, i], bin_preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh
    optimal_thresholds.append(best_thresh)
    bin_preds = (all_preds[:, i] > best_thresh).astype(int)
    rec = recall_score(all_true[:, i], bin_preds, zero_division=0)
    prec = precision_score(all_true[:, i], bin_preds, zero_division=0)
    print(f"{disease}: F1={best_f1:.4f}, Recall={rec:.4f}, Precision={prec:.4f}, Threshold={best_thresh:.2f}")

print("\nUse these thresholds for your final predictions.")

# === Step 11: Save model and scaler ===
torch.save(model.state_dict(), 'tcn_cattle_model_focal.pth')

import joblib
joblib.dump(scaler, 'feature_scaler1.save')

print("Model and scaler saved successfully.")


Sequence data shapes: X=(1929250, 24, 9), Y=(1929250, 4)
Epoch 1/20 - Training Loss: 0.30595
Epoch 2/20 - Training Loss: 0.29633
Epoch 3/20 - Training Loss: 0.29470
Epoch 4/20 - Training Loss: 0.29299
Epoch 5/20 - Training Loss: 0.29205
Epoch 6/20 - Training Loss: 0.29107
Epoch 7/20 - Training Loss: 0.29078
Epoch 8/20 - Training Loss: 0.29019
Epoch 9/20 - Training Loss: 0.28923
Epoch 10/20 - Training Loss: 0.28922
Epoch 11/20 - Training Loss: 0.28895
Epoch 12/20 - Training Loss: 0.28795
Epoch 13/20 - Training Loss: 0.28720
Epoch 14/20 - Training Loss: 0.28734
Epoch 15/20 - Training Loss: 0.28716
Epoch 16/20 - Training Loss: 0.28640
Epoch 17/20 - Training Loss: 0.28678
Epoch 18/20 - Training Loss: 0.28626
Epoch 19/20 - Training Loss: 0.28556
Epoch 20/20 - Training Loss: 0.28493

Per-disease threshold tuning and metrics:
oestrus: F1=0.0447, Recall=0.2481, Precision=0.0246, Threshold=0.59
calving: F1=0.0991, Recall=0.5415, Precision=0.0545, Threshold=0.59
lameness: F1=0.0147, Recall=0.601